In [ ]:
# papermill parameters cell — injected at runtime
run_id = ""
data_dir = ""
out_dir = ""
area_um2 = None
thickness_nm = None
device_id = ""

In [ ]:
import glob
import json
import os
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out_path = Path(out_dir)
out_path.mkdir(parents=True, exist_ok=True)
data_path = Path(data_dir)

EPS0 = 8.854e-12  # F/m

In [ ]:
# Find CF (C-F sweep) and CV (C-V sweep) CSV files
cf_files = sorted(data_path.glob('*_cf_*frequency_log*.csv'))
cv_files = sorted(data_path.glob('*_cv_*bias_linear*.csv'))

# Fallback: any cf/cv in name
if not cf_files:
    cf_files = sorted(data_path.glob('*cf*.csv'))
if not cv_files:
    cv_files = sorted(data_path.glob('*cv*.csv'))

print(f"CF files: {[f.name for f in cf_files]}")
print(f"CV files: {[f.name for f in cv_files]}")

if not cf_files and not cv_files:
    raise FileNotFoundError(f"No CF or CV CSV files found in {data_dir}")

In [ ]:
df_cf = None
if cf_files:
    df_cf = pd.read_csv(cf_files[0])
    print(f"CF: {len(df_cf)} rows — {list(df_cf.columns[:8])}")
    # Filter rows where frequency > 0 and C is valid
    if 'set_frequency_Hz' in df_cf.columns:
        df_cf = df_cf[df_cf['set_frequency_Hz'] > 0].copy()
    elif 'measured_frequency_Hz' in df_cf.columns:
        df_cf = df_cf[df_cf['measured_frequency_Hz'] > 0].copy()
    if 'C_F' in df_cf.columns:
        df_cf = df_cf[df_cf['C_F'].notna() & (df_cf['C_F'] > 0)].copy()
    print(f"CF after filter: {len(df_cf)} valid rows")

In [ ]:
df_cv = None
if cv_files:
    df_cv = pd.read_csv(cv_files[0])
    print(f"CV: {len(df_cv)} rows — {list(df_cv.columns[:8])}")
    # Keep only rows with valid C
    if 'C_F' in df_cv.columns:
        df_cv = df_cv[df_cv['C_F'].notna()].copy()
    print(f"CV after filter: {len(df_cv)} valid rows")

In [ ]:
cf_plot_path = None
if df_cf is not None and len(df_cf) > 0:
    freq_col = 'measured_frequency_Hz' if 'measured_frequency_Hz' in df_cf.columns else 'set_frequency_Hz'
    freq = df_cf[freq_col].to_numpy(dtype=float)
    cap = df_cf['C_F'].to_numpy(dtype=float)
    res = df_cf['R_Ohm'].to_numpy(dtype=float) if 'R_Ohm' in df_cf.columns else None

    n_plots = 2 if res is not None else 1
    fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5))
    if n_plots == 1:
        axes = [axes]

    ax = axes[0]
    ax.semilogx(freq, cap * 1e12, 'b-o', markersize=3, linewidth=1.2)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Capacitance (pF)')
    ax.set_title(f'C-F sweep\n{device_id or run_id[:16]}')
    ax.grid(True, which='both', alpha=0.3)

    if res is not None:
        ax2 = axes[1]
        ax2.semilogx(freq, res / 1e3, 'r-o', markersize=3, linewidth=1.2)
        ax2.set_xlabel('Frequency (Hz)')
        ax2.set_ylabel('Resistance (kΩ)')
        ax2.set_title(f'R-F sweep\n{device_id or run_id[:16]}')
        ax2.grid(True, which='both', alpha=0.3)

    fig.suptitle(f'C-F Impedance — run {run_id[:20]}', fontsize=11)
    fig.tight_layout()
    cf_plot_path = out_path / 'cf_sweep.png'
    fig.savefig(str(cf_plot_path), dpi=120, format='png')
    plt.close(fig)
    print(f"Saved CF plot: {cf_plot_path}")

In [ ]:
cv_plot_path = None
if df_cv is not None and len(df_cv) > 0:
    bias_col = 'set_bias_V' if 'set_bias_V' in df_cv.columns else df_cv.columns[0]
    bias = df_cv[bias_col].to_numpy(dtype=float)
    cap = df_cv['C_F'].to_numpy(dtype=float)
    res = df_cv['R_Ohm'].to_numpy(dtype=float) if 'R_Ohm' in df_cv.columns else None

    n_plots = 2 if res is not None else 1
    fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5))
    if n_plots == 1:
        axes = [axes]

    ax = axes[0]
    ax.plot(bias, cap * 1e12, 'b-o', markersize=3, linewidth=1.2)
    ax.set_xlabel('Bias Voltage (V)')
    ax.set_ylabel('Capacitance (pF)')
    ax.set_title(f'C-V sweep\n{device_id or run_id[:16]}')
    ax.grid(True, alpha=0.3)

    if res is not None:
        ax2 = axes[1]
        ax2.plot(bias, res / 1e3, 'r-o', markersize=3, linewidth=1.2)
        ax2.set_xlabel('Bias Voltage (V)')
        ax2.set_ylabel('Resistance (kΩ)')
        ax2.set_title(f'R-V sweep\n{device_id or run_id[:16]}')
        ax2.grid(True, alpha=0.3)

    fig.suptitle(f'C-V Impedance — run {run_id[:20]}', fontsize=11)
    fig.tight_layout()
    cv_plot_path = out_path / 'cv_sweep.png'
    fig.savefig(str(cv_plot_path), dpi=120, format='png')
    plt.close(fig)
    print(f"Saved CV plot: {cv_plot_path}")

In [ ]:
# Compute eps_r if thickness and area known
eps_r_accumulation = None
C_max = None

# Use CV data for eps_r (accumulation = max |C|)
ref_df = df_cv if df_cv is not None and len(df_cv) > 0 else df_cf
if ref_df is not None and 'C_F' in ref_df.columns:
    c_vals = ref_df['C_F'].to_numpy(dtype=float)
    c_valid = c_vals[~np.isnan(c_vals)]
    if len(c_valid) > 0:
        C_max = float(np.max(np.abs(c_valid)))
        if thickness_nm is not None and area_um2 is not None and area_um2 > 0:
            # eps_r = C * d / (eps0 * A)
            d_m = float(thickness_nm) * 1e-9
            A_m2 = float(area_um2) * 1e-12
            eps_r_accumulation = C_max * d_m / (EPS0 * A_m2)

print(f"C_max={C_max}, eps_r={eps_r_accumulation}")

In [ ]:
metrics = {
    'C_max_F': C_max,
    'eps_r_accumulation': eps_r_accumulation,
    'area_um2': area_um2,
    'thickness_nm': thickness_nm,
    'n_cf_points': int(len(df_cf)) if df_cf is not None else 0,
    'n_cv_points': int(len(df_cv)) if df_cv is not None else 0,
}

metrics_path = out_path / 'metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"Metrics: {metrics}")